This notebook produces & saves an scMPRA data object of the cohen retina scmpra experiment, using the ==Single counting U6 approach==. See below for details. 

# Setup

In [1]:
import scMPRAforge as scm
import pandas as pd
import itertools

2025-10-17 16:52:59.071275: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-17 16:52:59.075434: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

Load data

In [2]:
HEADWATER="/gpfs/gibbs/pi/reilly/tabula_data/cohen"
#raw_mpra=pd.read_csv(f"{HEADWATER}/read_wise_mpra_retina.tsv",sep="\t")
raw_u6=pd.read_csv(f"{HEADWATER}/unjoined/u6.tsv",sep="\t")

In [3]:
raw_u6

,reads,umi,pBC,rep_id,cell_bc,cell_type,cre_id
0,34,AAGGCGAGAGAG,ATGGAATG,1,AAACCCATCGCCGAGT,Rod,tail_conserve6_483_to_488
1,32,ACGCATCTCTTG,CGGTGAGA,1,AAACCCAGTAATCAGA,Rod,combo_Q50_crx4_crx2
2,32,ACTCTACCATAA,AATCTCCA,1,AAACCCAGTAATCAGA,Rod,ebox_gcagctgg_to_gcagctag
3,1,AGCCTCACAACC,ATGGAATG,1,AAACCCAGTAATCAGA,Rod,tail_conserve6_483_to_488
4,22,AGCCTCACAACG,ATGGAATG,1,AAACCCAGTAATCAGA,Rod,tail_conserve6_483_to_488
...,...,...,...,...,...,...,...
3640103,1,GCAACTCACCAT,CCTGAGTT,2,TTTGTTGAGCTGCCAC,Rod,combo_Mute_crx3_crx1
3640104,1,GTCTGAGGATCC,AGTGATTC,2,TTTGTTGAGCTGCCAC,Rod,tail_conserve4_428_to_433
3640105,26,GTGACTTTCTGT,CGCGTTAC,2,TTTGTTGAGCTGCCAC,Rod,combo_Q50_crx5_crx2
3640106,5,TTACAAGTCGCA,ATGGAATG,2,TTTGTTGAGCTGCCAC,Rod,tail_conserve6_483_to_488


In [4]:
mpra_dat=scm.scMPRA_data.from_tsv(f"{HEADWATER}/unjoined/read_wise_mpra_retina.tsv")

In [5]:
mpra_dat.read_wise_to_umi_wise()

In [6]:
mpra_dat.data

,cell_bc,rep_id,cre_id,cell_type,mpra_bc,umis
0,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,ACCTGTATGGAATGAGCGGGTCCA,1
1,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,ATTATAAAAGATACTCATGAGTCT,1
2,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,CGGTTGATTTCCGGTTTACCAGAT,1
3,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,GTGTTTAATCGTAGACGCGACGAT,1
4,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,GTTACCGAGGGCTGCAGTGTAGAG,1
...,...,...,...,...,...,...
7275510,TTTGTTGTCGTTCAGA,2,tail_conserve6_493_to_498,Rod,GGCTGAAACCACGTTCCTAATCTC,1
7275511,TTTGTTGTCGTTCAGA,2,tail_conserve6_498_to_503,Rod,CAACAGTTACCCTGCATACCTCGG,1
7275512,TTTGTTGTCGTTCAGA,2,tail_conserve6_498_to_503,Rod,TTGTAAGTCAGAATCTCATACCCC,1
7275513,TTTGTTGTCGTTCAGA,2,wt_1,Rod,TATTTGCCTATTCTTACATCACCT,1


# Rationale

How exactly do we want to handle integration of regular MPRA & tfection reporter?

So the thing is that the u6 data does not correspond directly to the MPRA bc. Instead, it corresponds to the CRE (pBC)...

Here's one approach: "Utilize U6 approach"
- For each replicate
  - For all combinations of cell barcode, CRE
    - If the CRE has at least one MPRA bc in that cell, move on. U6 info can tell us nothing about missing zeroes
    - If the CRE does **not** have at least one MPRA bc in that cell, check U6
      - If U6 says the CRE is present in that cell, add zeroes for all MPRAbc for that CRE in that cell
      - If the U6 does NOT say the CRE is present in that cell, do NOT add a zero.

The advantage of this approach is that it lets us use the U6 data to add in some true zeroes.

Disadvantage : A single transfected MPRA barcode in a given cell prevents U6 information from adding true zeroes to any other barcodes, causing undersetimation of zeroes.

Here's another approach: "Simple zero padding approach"
- For each replicate
  - For all combinations of cell barcode, MPRA barcode
    - if the combination is not present, add it as a zero
    - if the combination is present, continue.

The basic *problem* with this approach is that it over-estimates zeroes due to incomplete transfection. 

The advantages of this approach are that it is very simple and can be applied to datasets with NO transfection reporter. 

Note that the "Utilize U6 approach" can be considered a subset of the "simple zero padding approach". If we take "simple zero padding approach" and then simply remove those zeroes which are not supported by U6, we get "Utilize U6 approach". ~~For that reason, I'm going to briefly run both.~~ 

~~Actually....~~
~~there is perhaps a rather less intense way to get at the "utilize U6 approach" as listing all combos to start is maybe too intense... Here's an alternative approach~~
- ~~`rbc_map` = Create a map of CRE->rBC from the observed data. Replicate-agnostic.~~
- ~~missing = Anti-merge U6 with normal MPRA data to get all (CRE, cell bc, repid) combos which are NOT present in regular MPRA data~~
- ~~zero df = left merge missing, map on CRE id~~
- ~~final df = original MPRA df concat with zero df~~

Ok, here's another option. In [the original paper](https://www.nature.com/articles/s41588-022-01278-7), activity is calculated counting each case of U6 but no MPRA as one zero. 
=="Single-counting U6 approach"==
- u6_umi = summarize u6 umi counts
- padded = outer join of u6 & mpra data on cell_bc, rep_id, cre_id
- Then pad missing values


# Processing

In [7]:
u6_umi=raw_u6
u6_umi=u6_umi.drop(columns=["reads", "pBC"])
u6_umi=u6_umi.groupby([i for i in u6_umi.columns if i !="umi"]).aggregate(umis_transfection_bc= ('umi', 'nunique')).reset_index()
u6_umi

,rep_id,cell_bc,cell_type,cre_id,umis_transfection_bc
0,1,AAACCCAAGACAAGCC,Rod,combo_Mute_crx5_crx2,1
1,1,AAACCCAAGACAAGCC,Rod,combo_Q50_crx2,2
2,1,AAACCCAAGACAAGCC,Rod,ebox_gcagctgg_to_gcagcggg,1
3,1,AAACCCAAGACAAGCC,Rod,tail_conserve2_188_to_193,2
4,1,AAACCCAAGACAAGCC,Rod,tail_conserve3_240_to_245,1
...,...,...,...,...,...
159186,2,TTTGTTGTCGTTAGAC,Interneuron,tail_conserve3_215_to_220,1
159187,2,TTTGTTGTCGTTCAGA,Rod,combo_Mute_crx2,73
159188,2,TTTGTTGTCGTTCAGA,Rod,swap_crx2_to_gggcttag_reverse,2
159189,2,TTTGTTGTCGTTCAGA,Rod,tail_conserve2_193_to_198,2


In [8]:
padded=mpra_dat.data.merge(u6_umi,how="outer",indicator=True,on=["rep_id","cell_bc","cre_id"],validate="many_to_one")
#many to one since multiple rBC
padded

,cell_bc,rep_id,cre_id,cell_type_x,mpra_bc,umis,cell_type_y,umis_transfection_bc,_merge
0,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,ACCTGTATGGAATGAGCGGGTCCA,1.0,NaN,NaN,left_only
1,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,ATTATAAAAGATACTCATGAGTCT,1.0,NaN,NaN,left_only
2,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,CGGTTGATTTCCGGTTTACCAGAT,1.0,NaN,NaN,left_only
3,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,GTGTTTAATCGTAGACGCGACGAT,1.0,NaN,NaN,left_only
4,AAACCCAAGACAAGCC,1,combo_Mute_crx1,Rod,GTTACCGAGGGCTGCAGTGTAGAG,1.0,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...
7351464,TTTGTTGTCGTTCAGA,2,tail_conserve6_493_to_498,Rod,GGCTGAAACCACGTTCCTAATCTC,1.0,NaN,NaN,left_only
7351465,TTTGTTGTCGTTCAGA,2,tail_conserve6_498_to_503,Rod,CAACAGTTACCCTGCATACCTCGG,1.0,NaN,NaN,left_only
7351466,TTTGTTGTCGTTCAGA,2,tail_conserve6_498_to_503,Rod,TTGTAAGTCAGAATCTCATACCCC,1.0,NaN,NaN,left_only
7351467,TTTGTTGTCGTTCAGA,2,wt_1,Rod,TATTTGCCTATTCTTACATCACCT,1.0,NaN,NaN,left_only


In [9]:
padded["_merge"].value_counts()

_merge
left_only     4445565
both          2829950
right_only      75954
Name: count, dtype: int64

left_only means mpra but no u6, right_only means u6 but no MPRA, both means both. So we are only adding 75954 zeroes.

Let us fill in the missing values...

In [10]:
filled=padded

Let's fill in the cell-type information. First, make sure there are no conflicts.

In [11]:
conflicts = filled[
    filled["cell_type_x"].notna() & 
    filled["cell_type_y"].notna() & 
    (filled["cell_type_x"] != filled["cell_type_y"])
]
if not conflicts.empty:
    raise ValueError(f"Conflicting cell types found:\n{conflicts}")

Perfect. Now let's fill...

In [12]:
filled["cell_type"] = filled["cell_type_x"].combine_first(filled["cell_type_y"])
filled["umis"]=filled["umis"].fillna(0)
filled["umis_transfection_bc"]=filled["umis_transfection_bc"].fillna(0)
filled["mpra_bc"]=filled["mpra_bc"].fillna("dummy")


drop useless columns

In [14]:
filled=filled.drop(columns=['cell_type_x', 'cell_type_y','_merge'])

In [16]:
filled=filled.rename({'umis':'umis_mpra_bc'},axis=1)

Check if there are any NA...

In [17]:
filled.isnull().values.any()

False

Ok perfect! now let's dump this dataframe to disc...

In [18]:
filled.to_csv(f"{HEADWATER}/retina_single_counting_u6.tsv",sep="\t")

And make & save a scmpra object...

In [19]:
finalized_object=scm.scMPRA_data.from_tsv(f"{HEADWATER}/retina_single_counting_u6.tsv")

In [20]:
finalized_object.to_parquet(f"{HEADWATER}/retina_single_counting_u6.scmpra")